In [1]:
import torch
import torch.nn as nn

In [2]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [3]:
inputs.shape

torch.Size([6, 3])

In [18]:
class CausalAttention(nn.Module):
    def __init__(self, din, dout, context_length):
        super().__init__()
        self.dout = dout
        self.W_Key = nn.Linear(din, dout, bias=False)
        self.W_Query = nn.Linear(din, dout, bias=False)
        self.W_Value = nn.Linear(din, dout, bias=False)
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
    def forward(self, x):
        b, num_tokens, din = x.shape
        keys = self.W_Key(x)
        queries = self.W_Query(x)
        values = self.W_Value(x)
        
        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        
        context_vector = attn_weights @ values
        return context_vector

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, din, dout, num_heads, context_length):
        super().__init__()
        self.dout = dout
        self.num_heads = num_heads
        self.head_dim = dout // num_heads
        
        self.W_Key = nn.Linear(din, dout, bias=False)
        self.W_Query = nn.Linear(din, dout, bias=False)
        self.W_Value = nn.Linear(din, dout, bias=False)
        
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x):
        b, num_tokens, din = x.shape
        keys = self.W_Key(x)
        queries = self.W_Query(x)
        values = self.W_Value(x)
        
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = (attn_weights @ values).transpose(1,2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.dout)
        
        return context_vec


In [5]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) 
din = batch.shape[2]
dout = 6
context_length = batch.shape[1]


torch.Size([2, 6, 3])


In [6]:
mha = MultiHeadAttention(din, dout, 2, context_length)
context_vec = mha(batch)
print(context_vec.shape)
print(context_vec)

torch.Size([2, 6, 6])
tensor([[[-0.2577,  0.0958, -0.2736, -0.1411, -0.3326, -0.0127],
         [-0.0674, -0.0592, -0.1260, -0.2015, -0.5026,  0.0528],
         [-0.0070, -0.1049, -0.0835, -0.2261, -0.5597,  0.0813],
         [ 0.0242, -0.1238, -0.0404, -0.2007, -0.5274,  0.0718],
         [ 0.0186, -0.0752, -0.0907, -0.2325, -0.4753,  0.1316],
         [ 0.0370, -0.1099, -0.0431, -0.2011, -0.4881,  0.0928]],

        [[-0.2577,  0.0958, -0.2736, -0.1411, -0.3326, -0.0127],
         [-0.0674, -0.0592, -0.1260, -0.2015, -0.5026,  0.0528],
         [-0.0070, -0.1049, -0.0835, -0.2261, -0.5597,  0.0813],
         [ 0.0242, -0.1238, -0.0404, -0.2007, -0.5274,  0.0718],
         [ 0.0186, -0.0752, -0.0907, -0.2325, -0.4753,  0.1316],
         [ 0.0370, -0.1099, -0.0431, -0.2011, -0.4881,  0.0928]]],
       grad_fn=<ViewBackward0>)
